In [8]:
import pandas as pd
import numpy as np
import io
import os
from tqdm import tqdm
from curl_cffi import requests as cureq
from IPython.display import clear_output
import belo_horizonte_real_estate_market.functions.download_data as download_data

In [2]:
pd.set_option('display.max_columns', None)

In [3]:
dict_datasets_id = {"itbi": "0e13bf71-5355-47ce-8607-966413b08c0a",
                    "enderecamento": "394da7d4-3e74-4a2d-8fb6-6cb4d65d3451",
                    "cadastro_imobiliario_reg_pampulha": "e4f8fc31-df75-4f0b-9aab-f1abe1512d3f",
                    "cadastro_imobiliario_reg_oeste": "da432bf7-dcd4-463d-b10a-ab4922c8e2b3",
                    "cadastro_imobiliario_reg_norte": "df1cea37-92d0-4d52-917a-b1ed10bae4db",
                    "cadastro_imobiliario_reg_noroeste": "afb79a2b-69b3-4482-82d5-df3025070fd0",
                    "cadastro_imobiliario_reg_nordeste": "f7f03edc-39bc-4e17-aa94-a52d8bf56f04",
                    "cadastro_imobiliario_reg_barreiro": "41c80bc4-e0e3-4d06-9547-a8a1051c9fb6",
                    "cadastro_imobiliario_reg_leste": "f0745d2c-8b5c-4024-afb2-886c5de148e0",
                    "cadastro_imobiliario_reg_hipercentro": "4000eb81-51fc-47d7-931e-445099e0f45a",
                    "cadastro_imobiliario_reg_centrosul": "1ab07865-1a30-4e09-88c1-17d2183b2ea0",
                    "cadastro_imobiliario_reg_vendanova": "3ddf7e20-812d-4e74-99cd-814c942cb8c3",
                    "atividades_economicas_old": "3449df83-944b-4672-835d-d3e4a7bf7f48",
                    "atividades_economicas": "33e9dcb1-126f-4cde-8c80-d1927a965430",
                    "atividades_economicas_autonomos": "0e6890a3-6957-4d54-b6a6-0c77b97b1b88",
                    "edificacoes_licenciadas": "602b0331-286a-4a59-a987-6f6f38f6ebec",
                    "qtd_lancamentos_iptu_bairro": "177e1466-187d-47df-8638-1d02a060afc6",
                    "baixa_construcoes": "aa138ff1-7229-4101-ba1c-399eee7de8be"}

dict_tipo_construtivo = {"AP": "APARTAMENTO", "CA": "CASA", "SL": "SALA", "LV": "LOTE VAGO", "VC": "VAGA DE GARAGEM NAO RESIDENCIAL",
                         "LJ": "LOJA", "VR": "VAGA DE GARAGEM RESIDENCIAL", "BA": "BARRACAO", "GP": "GALPAO",
                         "AC": "APARTAMENTO COM OCUPACAO NAO RESIDENCIAL", "CC": "CASA COM OCUPACAO NAO RESIDENCIAL",
                         "BC": "BARRACAO COM OCUPACAO NAO RESIDENCIAL", "VV": "VAGA DE GARAGEM NAO RESIDENCIAL"}



In [4]:
list_resources = []
for key, value in tqdm(dict_datasets_id.items()):
    print(f"\n{key}")
    df_resource = download_data.list_files(value)\
    .assign(dataset = key)

    list_resources.append(df_resource)
    clear_output(wait = True)

df_resources = pd.concat(objs = list_resources, ignore_index = True)

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 18/18 [00:06<00:00,  2.96it/s]


In [6]:
df_itbi = df_resources\
.query("dataset == 'itbi' & format == 'CSV'")\
.apply(lambda df: download_data.get_csv_file(url = df["url"], dataset = "itbi"), axis = 1)

df_itbi = pd.concat(objs = list(df_itbi))\
.assign(area_terreno_total = lambda df: df["area_terreno_total"].str.replace(".", "").str.replace(",", ".").astype("float"))\
.assign(area_construida_adquirida = lambda df: df["area_construida_adquirida"].str.replace(".", "").str.replace(",", ".").astype("float"))\
.assign(area_adquirida_unidades_somadas = lambda df: df["area_adquirida_unidades_somadas"].str.replace(".", "").str.replace(",", ".").astype("float"))\
.assign(valor_declarado = lambda df: df["valor_declarado"].str.replace(".", "").str.replace(",", ".").astype("float"))\
.assign(valor_base_calculo = lambda df: df["valor_base_calculo"].str.replace(".", "").str.replace(",", ".").astype("float"))\
.assign(fracao_ideal_adquirida = lambda df: df["fracao_ideal_adquirida"].str.replace(",", ".").astype("float"))\
.assign(data_quitacao_transacao = lambda df: pd.to_datetime(df["data_quitacao_transacao"], format = "%d/%m/%Y"))\
.assign(ano_construcao_unidade = lambda df: [np.nan if i == 0 or i < 1800 and i > 2100 else i for i in df['ano_construcao_unidade']])\
.assign(tipo_construtivo_preponderante = lambda df: df["tipo_construtivo_preponderante"].map(dict_tipo_construtivo))\
.assign(urlfile = lambda df: df["urlfile"].str.split("/").apply(lambda x: x[-1]))

https://ckan.pbh.gov.br/dataset/0e13bf71-5355-47ce-8607-966413b08c0a/resource/7f8955aa-0b30-4157-bbc2-7dd444941728/download/pda_itbi_relatorio_200801_a_202405.csv: successful downloaded data!
https://ckan.pbh.gov.br/dataset/0e13bf71-5355-47ce-8607-966413b08c0a/resource/b21f9785-6b4c-43a1-bd66-dbf89e63efe7/download/pda_itbi_relatorio_202406.csv: successful downloaded data!
https://ckan.pbh.gov.br/dataset/0e13bf71-5355-47ce-8607-966413b08c0a/resource/1520a5dc-858a-4eca-89bd-54964c273ce7/download/pda_itbi_relatorio_202407_corrigido.csv: successful downloaded data!
https://ckan.pbh.gov.br/dataset/0e13bf71-5355-47ce-8607-966413b08c0a/resource/aa948bd0-9554-4e33-87f7-eb61ef0d2903/download/pda_itbi_relatorio_202408.csv: successful downloaded data!
https://ckan.pbh.gov.br/dataset/0e13bf71-5355-47ce-8607-966413b08c0a/resource/f582a331-9608-49a2-be3b-a5a5aa19ecff/download/pda_itbi_relatorio_202409.csv: successful downloaded data!
https://ckan.pbh.gov.br/dataset/0e13bf71-5355-47ce-8607-966413b08c

In [22]:
df_itbi.to_parquet(path = "../data/raw_itbi.parquet", engine = "fastparquet", compression = "zstd")
df_itbi

,endereco,bairro,ano_construcao_unidade,area_terreno_total,area_construida_adquirida,area_adquirida_unidades_somadas,padrao_acabamento_unidade,fracao_ideal_adquirida,tipo_construtivo_preponderante,descricao_tipo_ocupacao_unidade,valor_declarado,valor_base_calculo,zona_uso_itbi,data_quitacao_transacao,urlfile
0,AVE AFONSO PENA 3924 - GARAGE 60 - CRUZEIRO - ...,CRUZEIRO,1976.0,1119.00,28.53,28.53,P3,0.004470,VAGA DE GARAGEM NAO RESIDENCIAL,NAO RESIDENCIAL,1000.00,11411.56,ZA,2008-01-02,pda_itbi_relatorio_200801_a_202405.csv
1,AVE AMAZONAS 718 - APT 704 - CENTRO - 30180-00...,CENTRO,1960.0,1030.00,126.99,126.99,P2,0.007197,APARTAMENTO,RESIDENCIAL,85000.00,85000.00,ZHIP,2008-01-02,pda_itbi_relatorio_200801_a_202405.csv
2,AVE AUGUSTO DE LIMA 1276 - APT 301 - BARRO PRE...,BARRO PRETO,1978.0,544.00,135.55,135.55,P3,0.025843,APARTAMENTO,RESIDENCIAL,121500.00,121500.00,ZCBH,2008-01-02,pda_itbi_relatorio_200801_a_202405.csv
3,AVE AUGUSTO DE LIMA 1276 - GARAGE 14 - BARRO P...,BARRO PRETO,1978.0,544.00,11.45,11.45,P3,0.002182,VAGA DE GARAGEM RESIDENCIAL,RESIDENCIAL,13500.00,13500.00,ZCBH,2008-01-02,pda_itbi_relatorio_200801_a_202405.csv
4,AVE AUGUSTO DE LIMA 233 - SALA 1439 - CENTRO -...,CENTRO,1967.0,4426.00,25.20,25.20,P3,0.000450,SALA,NAO RESIDENCIAL,9041.00,10354.49,ZHIP,2008-01-02,pda_itbi_relatorio_200801_a_202405.csv
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2233,RUA VARGINHA 463 - BLOCO A APT 1503 - COLEGIO ...,COLEGIO BATISTA,1983.0,1578.00,107.00,107.00,P3,0.012202,APARTAMENTO,RESIDENCIAL,219072.01,472512.00,ZAP,2025-11-28,pda_itbi_relatorio_202511.csv
2234,AVE BIAS FORTES 1577 - GARAGE 14 - BARRO PRETO...,BARRO PRETO,1983.0,375.00,12.00,12.00,P3,0.003043,VAGA DE GARAGEM RESIDENCIAL,RESIDENCIAL,26863.85,35078.40,ZCBH,2025-11-30,pda_itbi_relatorio_202511.csv
2235,RUA HENRIQUE GORCEIX 2120 - BLOCO V APT 403 - ...,JARDIM MONTANHES,1990.0,4755.29,70.70,70.70,P2,0.008929,APARTAMENTO,RESIDENCIAL,294000.00,294000.00,ZAR2,2025-11-30,pda_itbi_relatorio_202511.csv
2236,RUA MARIA HEILBUTH SURETTE 1312 - APT 1201 - B...,BURITIS,2025.0,3308.00,194.48,194.48,P4,0.017478,APARTAMENTO,RESIDENCIAL,1245016.12,1245016.12,ZAR2,2025-11-30,pda_itbi_relatorio_202511.csv


In [27]:
df_cadastro_imobiliario = df_resources\
.query("dataset.str.contains('cadastro_imobiliario') & format == 'CSV'")\
.groupby("dataset")\
.apply(lambda df: df.iloc[[-1]], include_groups = False)\
.reset_index()\
.apply(lambda df: download_data.get_csv_file(url = df["url"]), axis = 1)

df_cadastro_imobiliario = pd.concat(objs = list(df_cadastro_imobiliario), ignore_index = True)\
.assign(urlfile = lambda df: df["urlfile"].str.split("/").apply(lambda x: x[-1]))\
.drop(columns = ["frequencia_coleta", "ind_meio_fio", "ind_pavimentacao", "ind_arborizacao", "ind_galeria_pluvial",
                 "ind_iluminacao_publica", "ind_rede_esgoto", "ind_rede_agua", "ind_rede_telefonica"])\
.astype(dtype = {"cep": "str", "numero_imovel": "str", "nulotctm": "str"})\
.assign(numero_imovel = lambda df: df["numero_imovel"].str.replace("\\.0", "", regex = True))\
.assign(cep = lambda df: df["cep"].str.replace("^0$", "", regex = True))

https://ckan.pbh.gov.br/dataset/41c80bc4-e0e3-4d06-9547-a8a1051c9fb6/resource/fe7d8191-35d4-4910-802a-5cf5da468fbd/download/20251103_regional_barreiro_cadastro_imobiliario.csv: successful downloaded data!
https://ckan.pbh.gov.br/dataset/1ab07865-1a30-4e09-88c1-17d2183b2ea0/resource/6d6e246d-357e-43fe-baa9-25a46f4e4326/download/20251103_regional_centro_sul_cadastro_imobiliario.csv: successful downloaded data!
https://ckan.pbh.gov.br/dataset/4000eb81-51fc-47d7-931e-445099e0f45a/resource/0757ce4f-dd06-4540-b93a-4d903b8d2f11/download/20251103_regional_hipercentro_cadastro_imobiliario.csv: successful downloaded data!
https://ckan.pbh.gov.br/dataset/f0745d2c-8b5c-4024-afb2-886c5de148e0/resource/3c9c7e85-69a0-47d5-bf6c-64b905223b60/download/20251103_regional_leste_cadastro_imobiliario.csv: successful downloaded data!
https://ckan.pbh.gov.br/dataset/f7f03edc-39bc-4e17-aa94-a52d8bf56f04/resource/045d3d12-2b2a-4194-a449-b26e2d66f6c7/download/20251103_regional_nordeste_cadastro_imobiliario.csv: s

In [29]:
df_cadastro_imobiliario.to_parquet(path = "../data/raw_cadastro_imobiliario.parquet", engine = "fastparquet", compression = "zstd")
df_cadastro_imobiliario

,id_iptu_ctm,indice_cadastral,nulotctm,zoneamento_pviptu,area_terreno,area_construcao,tipo_construtivo,tipo_ocupacao,padrao_acabamento,quantidade_economias,fracao_ideal,tipo_logradouro,nome_logradouro,numero_imovel,cep,zona_homogenia,tipologia,geometria,urlfile
0,744,227031 005 001X,220384400270,ZEIS1,1140.00,0.00,LOTE VAGO,TERRITORIAL,TE,1,1.000000,RUA,MARIA DAS MERCES E SILVA,57,30662260,BA123,TERRITORIAL,"POLYGON ((600699.75 7789178.5,600685.1 7789194...",20251103_regional_barreiro_cadastro_imobiliari...
1,320,204064 015Z0160,120799900850,ZAR2,1771.07,66.57,APARTAMENTO,RESIDENCIAL,P3,1,0.036023,RUA,WILSON TAVARES RIBEIRO,690,30644260,BA224,FRENTE,"POLYGON ((602689.44 7789505,602689.3 7789508.5...",20251103_regional_barreiro_cadastro_imobiliari...
2,10412,618158B045 0014,121081000525,ZE,161.28,0.00,LOTE VAGO,TERRITORIAL,TE,1,1.000000,RUA,VISTA DO ROLA MOCA,nan,,BA236,TERRITORIAL,"POLYGON ((603279.4 7786838.5,603286.2 7786827,...",20251103_regional_barreiro_cadastro_imobiliari...
3,10605,618158G025 0013,120950600275,ZEIS1,43.59,0.00,LOTE VAGO,TERRITORIAL,TE,1,1.000000,RUA,NOSSA SENHORA APARECIDA,1,30628610,BA234,NaN,"POLYGON ((603308.25 7787049.5,603306.06 778705...",20251103_regional_barreiro_cadastro_imobiliari...
4,10250,634050 022 0023,121089100160,ZAR2,515.00,160.56,CASA,RESIDENCIAL,P2,1,1.000000,RUA,DECIO DE OLIVEIRA SALES,100,30672620,BA133,FRENTE,"POLYGON ((601735.5 7785703,601732.8 7785732.5,...",20251103_regional_barreiro_cadastro_imobiliari...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
892536,819145,888025 002 0366,190404800115,ZCVN,987.20,32.87,SALA,NAO RESIDENCIAL,P4,1,0.012190,RUA,PADRE PEDRO PINTO,382,31610000,VN316,DEMAIS CASOS,"POLYGON ((609622.75 7808323.5,609622.25 780832...",20251103_regional_venda_nova_cadastro_imobilia...
892537,819149,863091 027 0054,210100800225,ZAP,480.00,124.88,APARTAMENTO,RESIDENCIAL,P2,1,0.259100,RUA,DOS MOICANOS,55,31530360,VN311,FRENTE,"POLYGON ((606543.9 7807185.5,606534.06 7807213...",20251103_regional_venda_nova_cadastro_imobilia...
892538,819153,971011 013 0090,210554400450,ZAP,360.00,112.98,APARTAMENTO,RESIDENCIAL,P3,1,0.124486,RUA,SEBASTIAO PATRUS DE SOUZA,95,31535100,VN305,FRENTE,"POLYGON ((606771.8 7808317,606769 7808317,6067...",20251103_regional_venda_nova_cadastro_imobilia...
892539,819324,975065 025 0090,210724700040,ZAP,368.43,86.64,APARTAMENTO,RESIDENCIAL,P3,1,0.073280,RUA,PEDRA DO MAR,444,31570170,VN304,FRENTE,"POLYGON ((606612.75 7809032.5,606611.25 780903...",20251103_regional_venda_nova_cadastro_imobilia...


In [ ]:
df_resource = df_resources\
.query("dataset == 'enderecamento' & format == 'CSV'")\
.reset_index()\
.sort_values("index", ascending = False)\
.reset_index(drop = True)

df_enderecamento = pd.DataFrame()
for index in tqdm(range(df_resource.shape[0])):
    df_ = download_data.get_csv_file(url = df_resource.iloc[index]["url"])
    df_enderecamento = pd.concat(objs = [df_enderecamento, df_], ignore_index = True)\
    .drop_duplicates()


  0%|                                                                                                                                           | 0/23 [00:00<?, ?it/s]

https://ckan.pbh.gov.br/dataset/394da7d4-3e74-4a2d-8fb6-6cb4d65d3451/resource/865b565d-17c3-44a4-87ab-16a523b7f75d/download/20251103_endereco.csv: successful downloaded data!


  4%|█████▋                                                                                                                             | 1/23 [00:15<05:35, 15.26s/it]

https://ckan.pbh.gov.br/dataset/394da7d4-3e74-4a2d-8fb6-6cb4d65d3451/resource/c02012bb-5121-43b5-957d-cf09fff2c702/download/20251001_endereco.csv: successful downloaded data!


  9%|███████████▍                                                                                                                       | 2/23 [00:32<05:46, 16.50s/it]

...

https://ckan.pbh.gov.br/dataset/394da7d4-3e74-4a2d-8fb6-6cb4d65d3451/resource/95bdf449-55bd-4548-921f-ec652b3b2c71/download/20250204_endereco.csv: successful downloaded data!


 43%|████████████████████████████████████████████████████████▌                                                                         | 10/23 [06:23<11:14, 51.85s/it]

https://ckan.pbh.gov.br/dataset/394da7d4-3e74-4a2d-8fb6-6cb4d65d3451/resource/5238774d-70ad-423e-9773-e06b5bf46f96/download/20250102_endereco.csv: successful downloaded data!


 48%|██████████████████████████████████████████████████████████████▏                                                                   | 11/23 [07:29<11:14, 56.19s/it]

https://ckan.pbh.gov.br/dataset/394da7d4-3e74-4a2d-8fb6-6cb4d65d3451/resource/6c670cc7-55df-46b1-ad12-2a46f14a693c/download/20241129_endereco.csv: successful downloaded data!


In [20]:


# df_enderecamento = df_resources\
# .query("dataset == 'enderecamento' & format == 'CSV'")\
# .groupby("dataset")\
# .apply(lambda df: df.iloc[[-1]], include_groups = False)\
# .reset_index()\
# .apply(lambda df: get_csv_file(url = df["url"]), axis = 1)

# df_enderecamento = pd.concat(objs = list(df_enderecamento), ignore_index = True)\
# .astype(dtype = {"cep": "str", "numero_imovel": "str"})\
# .assign(numero_imovel = lambda df: df["numero_imovel"].str.replace("\\.0", "", regex = True))\
# .assign(cep = lambda df: df["cep"].str.replace("\\.0", "", regex = True))\
# .assign(urlfile = lambda df: df["urlfile"].str.split("/").apply(lambda x: x[-1]))